# Lab 2 Exercise

The REST walkthrough already showed the building blocks. In this exercise, those building blocks are provided so you can focus on detecting failure and rerouting.

> **Run cells in order from top to bottom.** Later exercises depend on variables set by earlier cells. Jumping ahead will cause `NameError` or stale state.

<details>
<summary>New to Jupyter? Click here</summary>

- **Run a cell**: click the cell, then press **Shift+Enter** (or click the ▶ Run button in the toolbar)
- **Cell state**: a number like `[3]` means the cell has run; `[*]` means it is still running
- **Cells share state**: variables set in one cell are available in all cells below it — this is why order matters
- **Stop a long-running cell**: click the **■ Stop** button in the toolbar, or go to **Kernel → Interrupt**
- **Something broke**: go to **Kernel → Restart & Clear Output**, then re-run every cell from the top

</details>

## What this exercise is about

In the walkthrough, ONOS was still helping you because `org.onosproject.fwd` could install forwarding rules for new traffic.

In this exercise, you will turn that app off and replace that behavior with your own notebook code.

Your notebook will act like a tiny controller app:

1. install rules for the current path between `h1` and `h2`
2. watch the links on that path
3. detect when one of those links fails
4. remove the old rules and install new ones on the alternate path

<img src="triangle-reroute.svg" alt="Automatic reroute overview" width="620">

## Before you start

Keep Mininet, ONOS CLI, and Jupyter running.

If the ONOS CLI is not open yet, start it in a terminal with:

```text
ssh -p 8101 -o HostKeyAlgorithms=+ssh-rsa onos@localhost
```

In the ONOS CLI, run:

```text
app deactivate org.onosproject.fwd
```

Then check that the old rules are gone:

```text
flows
```

Wait until no `org.onosproject.fwd` rules remain.

In the Mininet CLI, confirm connectivity is now broken:

```text
h1 ping -c 1 h2
```

That ping should fail. If it does not, the `fwd` app is still active — deactivate it and wait a few seconds before retrying.

In [ ]:
import json
import requests
import time

## Provided code

Run the next cell. It defines all the helper functions you will use below.

These helpers are thin wrappers around the same ONOS REST calls and flow-rule payloads you already worked with in `rest_walkthrough.ipynb`.

You do not need to edit any of them, but split your attention like this:

- Background wrappers you can mostly treat as already-built plumbing: `api_get(...)`, `api_post(...)`, `api_delete(...)`, `find_host_by_ip(...)`, `get_host_location(...)`, `build_flow_rule(...)`
- Helpers to pay specific attention to because later parts use or reason about them directly: `get_path(...)`, `install_path_rules(...)`, `remove_rules_by_app_id(...)`, `get_port_status(...)`

Run the cell first, then skim that second group so you know what each function is responsible for.

### Background wrappers

Run this cell. These helpers package up REST calls, host lookups, and flow-rule construction so you do not have to rewrite that boilerplate in the exercise.

In [ ]:
def api_get(endpoint, base, auth):
    """Fetch a JSON response from one ONOS REST endpoint."""
    response = requests.get(f'{base}/{endpoint}', auth=auth, timeout=5)
    response.raise_for_status()
    return response.json()


def api_post(endpoint, data, base, auth):
    """Send JSON data to one ONOS REST endpoint with POST."""
    response = requests.post(
        f'{base}/{endpoint}',
        auth=auth,
        json=data,
        timeout=5,
    )
    response.raise_for_status()
    return response


def api_delete(endpoint, base, auth):
    """Delete one ONOS REST resource and return the response."""
    response = requests.delete(f'{base}/{endpoint}', auth=auth, timeout=5)
    response.raise_for_status()
    return response


def find_host_by_ip(hosts, ip):
    """Return the discovered host record that owns the given IP address."""
    for host in hosts:
        if ip in host.get('ipAddresses', []):
            return host
    return None


def get_host_location(host):
    """Return the switch ID and port where a host is attached."""
    locations = host.get('locations', [])
    if not locations:
        return None, None
    location = locations[0]
    return location['elementId'], location['port']


def get_path(src_device, dst_device, base, auth):
    """Ask ONOS for the current path between two switches.

    Returns a list of link objects you can loop over later, or None if ONOS
    does not currently have a path.
    """
    data = api_get(f'paths/{src_device}/{dst_device}', base, auth)
    paths = data.get('paths', [])
    if not paths:
        return None
    return paths[0]['links']


def build_flow_rule(src_ip, dst_ip, out_port, app_id, priority=40000):
    """Build one IPv4 flow-rule payload for ONOS."""
    return {
        'priority': priority,
        'timeout': 0,
        'isPermanent': True,
        'appId': app_id,
        'treatment': {
            'instructions': [
                {'type': 'OUTPUT', 'port': str(out_port)}
            ]
        },
        'selector': {
            'criteria': [
                {'type': 'ETH_TYPE', 'ethType': '0x0800'},
                {'type': 'IPV4_SRC', 'ip': f'{src_ip}/32'},
                {'type': 'IPV4_DST', 'ip': f'{dst_ip}/32'},
            ]
        },
    }


### Helpers to pay attention to

Run this cell too. These are the helpers that later parts call directly or ask you to reason about when installing, removing, and checking path rules.

In [ ]:


def install_path_rules(
    path_links,
    src_ip,
    dst_ip,
    src_host_port,
    dst_host_port,
    base,
    auth,
    app_id,
):
    """Install end-to-end rules for the current path.

    path_links describes the switch-to-switch part of the route, and
    src_host_port and dst_host_port add the edge rules that connect the
    hosts onto that path. The function installs rules in both directions so
    h1->h2 and h2->h1 traffic both work.
    """
    first_device = path_links[0]['src']['device']
    last_device = path_links[-1]['dst']['device']

    for link in path_links:
        src_device = link['src']['device']
        src_port = link['src']['port']
        dst_device = link['dst']['device']
        dst_port = link['dst']['port']

        forward_rule = build_flow_rule(src_ip, dst_ip, src_port, app_id)
        reverse_rule = build_flow_rule(dst_ip, src_ip, dst_port, app_id)

        api_post(f'flows/{src_device}', forward_rule, base, auth)
        api_post(f'flows/{dst_device}', reverse_rule, base, auth)

    dst_edge_rule = build_flow_rule(src_ip, dst_ip, dst_host_port, app_id)
    src_edge_rule = build_flow_rule(dst_ip, src_ip, src_host_port, app_id)

    api_post(f'flows/{last_device}', dst_edge_rule, base, auth)
    api_post(f'flows/{first_device}', src_edge_rule, base, auth)


def remove_rules_by_app_id(device_ids, base, auth, app_id):
    """Remove this notebook's rules from the listed switches.

    It only deletes flows whose appId matches app_id, which is why rerouting
    and cleanup can safely remove our rules without touching rules from other
    ONOS apps.
    """
    removed = 0
    for device_id in device_ids:
        flows = api_get(f'flows/{device_id}', base, auth).get('flows', [])
        for flow in flows:
            if flow.get('appId') == app_id:
                api_delete(f"flows/{device_id}/{flow['id']}", base, auth)
                removed += 1
    return removed


def get_port_status(device_id, base, auth):
    """Return ONOS's current port records for one switch.

    detect_failure(...) uses this to look up the specific port from a path
    link and check whether ONOS still reports that port as enabled.
    """
    return api_get(f'devices/{device_id}/ports', base, auth)['ports']

## Ready check

Run the next cell before doing anything else.

You want to see:

- at least 2 hosts discovered
- `org.onosproject.fwd` reported as inactive
- 0 remaining `org.onosproject.fwd` rules
- `[ready]` at the end

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

devices = api_get('devices', base, auth).get('devices', [])
hosts = api_get('hosts', base, auth).get('hosts', [])
applications = api_get('applications', base, auth).get('applications', [])
fwd_app = next((app for app in applications if app.get('name') == 'org.onosproject.fwd'), None)
fwd_state = fwd_app.get('state', 'unknown') if fwd_app else 'not found'
fwd_is_active = fwd_state == 'ACTIVE' or bool(fwd_app and (fwd_app.get('active') or fwd_app.get('isActive')))

fwd_rule_count = 0
for device in devices:
    flows = api_get(f"flows/{device['id']}", base, auth).get('flows', [])
    fwd_rule_count += sum(1 for flow in flows if flow.get('appId') == 'org.onosproject.fwd')

print(f'devices discovered: {len(devices)}')
print(f'hosts discovered: {len(hosts)}')
print(f'org.onosproject.fwd state: {fwd_state}')
print(f'org.onosproject.fwd rules remaining: {fwd_rule_count}')

if len(hosts) >= 2 and not fwd_is_active and fwd_rule_count == 0:
    print('[ready] You can continue.')
else:
    print('[not ready] Fix the checks above first.')

## Sanity check

Before you build rerouting, make sure the provided helpers can install the current path once.

Run the next cell.

Do not continue until it prints `[ready]`.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'
app_id = 'org.onosproject.rest'

device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
remove_rules_by_app_id(device_ids, base, auth, app_id)

hosts = api_get('hosts', base, auth)['hosts']
src_host = find_host_by_ip(hosts, src_ip)
dst_host = find_host_by_ip(hosts, dst_ip)
src_device, src_host_port = get_host_location(src_host)
dst_device, dst_host_port = get_host_location(dst_host)
path_links = get_path(src_device, dst_device, base, auth)

if not path_links:
    print('[not ready] No path found. Recheck ONOS discovery before continuing.')
else:
    install_path_rules(
        path_links,
        src_ip,
        dst_ip,
        src_host_port,
        dst_host_port,
        base,
        auth,
        app_id,
    )

    installed_rule_count = 0
    for device_id in device_ids:
        flows = api_get(f'flows/{device_id}', base, auth).get('flows', [])
        installed_rule_count += sum(1 for flow in flows if flow.get('appId') == app_id)

    print(json.dumps(path_links, indent=2))
    print(f'Rules installed for the current path: {installed_rule_count}')

    if installed_rule_count > 0:
        print('[ready] Notebook rules were installed. Now do the manual ping/flows check below.')
    else:
        print('[not ready] No rules with this notebook\'s appId were found after installation.')

**Verify**

In the Mininet CLI, run:

```text
h1 ping -c 3 h2
```

That ping should work — our code installed the rules.

In the ONOS CLI, run:

```text
flows
```

You should see rules with `appId=org.onosproject.rest`.

If either manual check fails, do not continue yet even if the notebook cell printed `[ready]`.

## Part 1: Find the switches on the current path

Complete `get_active_devices`.

**Why this matters**

Before rerouting, we need to know which switches have our old rules on them so we can clean them up. This function builds that set.

**What you are doing here**

`get_path(...)` gives you the current route as a list of links between switches. Each link tells you which switch the hop starts from (`src`) and which switch it goes to (`dst`).

In this part, you are walking through that list and collecting the switch IDs that appear anywhere on the path. We use a `set` because the same switch should only appear once in the final answer, even if it shows up in more than one link.

**Goal**

Return the set of device IDs touched by the current path.

**Hints**

- each `link` is a dictionary with nested `src` and `dst` dictionaries
- the device ID you want lives at `link['src']['device']` or `link['dst']['device']`
- a `set()` avoids duplicates if the same switch appears on more than one link
- add items to a set with `devices.add(...)`
- your final answer should look like a set of switch IDs such as `{'of:0000000000000001', 'of:0000000000000002'}`

**Optional peek**

If the nested structure feels unfamiliar, run the next cell to inspect one real link from `path_links` before you fill in the function.

When you look at the output, focus on just one question: which value names the source switch, and which value names the destination switch?

In [ ]:
print(json.dumps(path_links[0], indent=2))

In [ ]:
def get_active_devices(path_links):
    devices = set()

    for link in path_links:
        src_device = link['src']['device']
        dst_device = link['dst']['device']
        # TODO: add src_device to devices
        # TODO: add dst_device to devices

    return devices

<details>
<summary>Show solution</summary>

```python
def get_active_devices(path_links):
    devices = set()

    for link in path_links:
        src_device = link['src']['device']
        dst_device = link['dst']['device']
        devices.add(src_device)
        devices.add(dst_device)

    return devices
```

</details>

**Verify**

Run the next cell.

Do not continue until it prints `[ready]`.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

hosts = api_get('hosts', base, auth)['hosts']
src_host = find_host_by_ip(hosts, '10.0.0.1')
dst_host = find_host_by_ip(hosts, '10.0.0.2')
src_device, _ = get_host_location(src_host)
dst_device, _ = get_host_location(dst_host)
path_links = get_path(src_device, dst_device, base, auth)

active_devices = get_active_devices(path_links)
sorted_devices = sorted(active_devices)
print(sorted_devices)

if not path_links:
    print('[not ready] No current path was found.')
elif not isinstance(active_devices, set):
    print('[not ready] get_active_devices(...) should return a set.')
elif len(sorted_devices) < 2:
    print('[not ready] Expected at least two switches on the current path.')
elif not all(str(device_id).startswith('of:') for device_id in sorted_devices):
    print('[not ready] The result does not look like a set of switch IDs.')
else:
    print('[ready] Active devices look reasonable. You can continue.')

## Part 2: Detect whether the current path has failed

Complete `detect_failure`.

**Why this matters**

The monitor loop calls this every few seconds. It needs to notice the moment a port on our path goes down so rerouting can start immediately.

**What you are doing here**

For each link on the current path, you already know the source switch (`device_id`) and the outgoing port on that switch (`path_port`).

In this part, you ask ONOS for that switch's current port records, then look for the entry whose port number matches `path_port`. If that matching port is disabled, the path has failed and the function should return `True` immediately.

If you finish checking every link and never find a disabled path port, then the path is still healthy and the function should return `False`.

**Goal**

Return `True` if any port along the current path is down. Otherwise return `False`.

**Hints**

- `device_id` and `path_port` are already set above the TODO — use them
- `get_port_status(...)` returns a list of dictionaries; each one describes one port on that switch
- the fields you care about are `port['port']` and `port['isEnabled']`
- port numbers from the API can come back as integers, so cast `port['port']` to `str()` before comparing it with `path_port`
- once you find a matching path port that is disabled, return right away

**Optional peek**

If you want to see the exact data shape first, run the next cell.

It looks at the first link in `path_links`, asks ONOS for that source switch's ports, and prints the one port record that matches the path port. Before you bring a link down, its `isEnabled` value should normally be `true`.

In [ ]:
device_id = path_links[0]['src']['device']
path_port = str(path_links[0]['src']['port'])
ports = get_port_status(device_id, base, auth)
matching_ports = [port for port in ports if str(port['port']) == path_port]

print(f'Checking switch {device_id}, path port {path_port}')
print(json.dumps(matching_ports, indent=2))

In [ ]:
def detect_failure(path_links, base, auth):
    for link in path_links:
        device_id = link['src']['device']
        path_port = str(link['src']['port'])
        ports = get_port_status(device_id, base, auth)

        for port in ports:
            port_number = str(port['port'])
            is_enabled = port['isEnabled']
            # TODO: return True when port_number matches path_port and is_enabled is False

    return False

<details>
<summary>Show solution</summary>

```python
def detect_failure(path_links, base, auth):
    for link in path_links:
        device_id = link['src']['device']
        path_port = str(link['src']['port'])
        ports = get_port_status(device_id, base, auth)

        for port in ports:
            port_number = str(port['port'])
            is_enabled = port['isEnabled']
            if port_number == path_port and not is_enabled:
                return True

    return False
```

</details>

**Verify**

Run the next cell to save a snapshot of the current (healthy) path. **Do this before bringing the link down.**

This cell also checks that the saved path still looks healthy right now. Do not continue until it prints `[ready]`.

The variable `path_links_before_failure` is used by parts 3 and 4.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

hosts = api_get('hosts', base, auth)['hosts']
src_host = find_host_by_ip(hosts, '10.0.0.1')
dst_host = find_host_by_ip(hosts, '10.0.0.2')
src_device, _ = get_host_location(src_host)
dst_device, _ = get_host_location(dst_host)
path_links_before_failure = get_path(src_device, dst_device, base, auth)

if not path_links_before_failure:
    print('[not ready] No healthy path was found. Recheck ONOS discovery before continuing.')
else:
    print(json.dumps(path_links_before_failure, indent=2))
    if detect_failure(path_links_before_failure, base, auth):
        print('[not ready] This saved path already looks failed. Make sure the link is up before continuing.')
    else:
        print('[ready] Healthy path snapshot saved. Now bring the link down.')

Now go to the Mininet CLI and run:

```text
link s1 s2 down
```

Wait a few seconds for ONOS to notice the port state change, then run the next cell.

Do not continue to part 3 unless that cell prints `[ready]`. If it does not, either the link state has not propagated yet or `detect_failure(...)` still needs fixing.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')

if 'path_links_before_failure' not in globals():
    print('[not ready] No saved path snapshot found. Re-run the previous verify cell first.')
elif not path_links_before_failure:
    print('[not ready] The saved path snapshot is empty. Re-run the previous verify cell first.')
else:
    failure_detected = detect_failure(path_links_before_failure, base, auth)
    print(f'detect_failure(...) returned: {failure_detected}')

    if failure_detected:
        print('[ready] Failure detected correctly. You can continue to part 3.')
    else:
        print('[not ready] ONOS does not yet show the saved path as failed, or detect_failure(...) is still incomplete.')
        print('Re-run `link s1 s2 down` in Mininet if needed, wait a few seconds, and try this cell again.')

## Part 3: Reroute once after a failure

Complete `reroute_once`.

**Why this matters**

This is the core of a controller application: when a link fails, clean up the stale rules, ask ONOS for a new path, and install fresh rules on it. Everything else is just monitoring that calls this function.

**What you are doing here**

You already saved the old path before the failure in `path_links_before_failure`. That old path tells you which switches currently have this notebook's rules on them.

In this part, you do the reroute in three stages:

1. remove the old rules from the switches on the failed path
2. ask ONOS for the new path now that the link is down
3. install fresh rules along that new path and return it

If ONOS cannot find an alternate path, the function should return `None`.

**Goal**

Remove the old rules, query the new path, install new rules, and return the new path.

**Note** Keep the `s1-s2` link down from part 2. If you accidentally brought it back up, run `link s1 s2 down` in Mininet again before running the verify cell.

**Hints**

- revisit the **Helpers to pay attention to** section above; both TODOs are single helper-function calls from that section
- for the first TODO, look for the helper whose job is to remove this notebook's rules by `app_id`
- for the second TODO, look for the helper whose job is to install rules along a path using the host-edge ports
- the rest of the logic (fetching hosts, computing the new path, handling the no-path case) is already written for you

**Optional peek**

If you want to sanity-check the cleanup step first, run the next cell.

It shows which switches from the saved pre-failure path will have old rules removed during rerouting.

In [ ]:
if 'path_links_before_failure' not in globals() or not path_links_before_failure:
    print('[not ready] Save a healthy path in part 2 first.')
else:
    print(sorted(get_active_devices(path_links_before_failure)))

In [ ]:
def reroute_once(path_links, src_ip, dst_ip, base, auth, app_id):
    device_ids = sorted(get_active_devices(path_links))
    # TODO: In the "Helpers to pay attention to" section, find the cleanup helper
    # that removes rules by app_id. Use it here with device_ids.
    time.sleep(2)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    new_path_links = get_path(src_device, dst_device, base, auth)

    if not new_path_links:
        return None

    # TODO: In the same helper section, find the helper that installs rules for a path.
    # Use it here with new_path_links, the host ports, and app_id.
    return new_path_links

<details>
<summary>Show solution</summary>

```python
def reroute_once(path_links, src_ip, dst_ip, base, auth, app_id):
    device_ids = sorted(get_active_devices(path_links))
    remove_rules_by_app_id(device_ids, base, auth, app_id)
    time.sleep(2)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    new_path_links = get_path(src_device, dst_device, base, auth)

    if not new_path_links:
        return None

    install_path_rules(
        new_path_links,
        src_ip,
        dst_ip,
        src_host_port,
        dst_host_port,
        base,
        auth,
        app_id,
    )
    return new_path_links
```

</details>

**Verify**

The `s1-s2` link should still be down from the previous step.

Run the next cell. It will call `reroute_once`, print the new path, and check whether rerouting appears to have succeeded.

Do not continue until it prints `[ready]`.

This check looks for two things: ONOS should report a different path, and the notebook should have installed its own rules on the newly involved switch(es).

Then in Mininet run:

```text
h1 ping -c 3 h2
```

That ping should work again — traffic is now going via the alternate hop. In the ONOS CLI:

```text
flows
```

You should see rules with `appId=org.onosproject.rest`.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'
app_id = 'org.onosproject.rest'

if 'path_links_before_failure' not in globals() or not path_links_before_failure:
    print('[not ready] No saved pre-failure path found. Re-run the part 2 verify step first.')
elif not detect_failure(path_links_before_failure, base, auth):
    print('[not ready] The saved path does not currently look failed. Make sure `link s1 s2 down` is still in effect.')
else:
    new_path_links = reroute_once(path_links_before_failure, src_ip, dst_ip, base, auth, app_id)

    if not new_path_links:
        print('[not ready] ONOS did not return an alternate path.')
    else:
        print(json.dumps(new_path_links, indent=2))

        old_active_devices = get_active_devices(path_links_before_failure)
        new_active_devices = get_active_devices(new_path_links)
        new_only_device_ids = sorted(new_active_devices - old_active_devices)

        notebook_rules_on_new_only = 0
        for device_id in new_only_device_ids:
            flows = api_get(f'flows/{device_id}', base, auth).get('flows', [])
            notebook_rules_on_new_only += sum(1 for flow in flows if flow.get('appId') == app_id)

        print(f'Newly involved switches: {new_only_device_ids}')
        print(f'Notebook rules on newly involved switches: {notebook_rules_on_new_only}')

        if new_path_links == path_links_before_failure:
            print('[not ready] The returned path is still the same as the old saved path.')
        elif not new_only_device_ids:
            print('[not ready] The verify step could not identify any newly involved switch to confirm reroute installation.')
        elif notebook_rules_on_new_only == 0:
            print('[not ready] ONOS found a different path, but this notebook did not install any rules on the newly involved switch(es).')
        else:
            print('[ready] Reroute completed and notebook rules were installed on the new path. Now verify connectivity with ping.')

## Milestone

If you reached this point, you have already built the core controller behavior.

Your notebook can now:

1. identify the switches on the active path
2. detect when a path port goes down
3. remove stale rules and install rules for a new path

That is the hard part.

In SDN terms, the switches are still handling the data plane, but your notebook is already acting like a small control-plane application: it observes state and decides what rules should exist.

Part 4 just wraps that logic in a loop so the notebook keeps watching and reroutes automatically whenever a future failure appears.

## Part 4: Build the automatic loop

Complete `monitor_and_reroute`.

**Why this matters**

`reroute_once` handles a single failure. This function wraps it in a poll loop so the app keeps watching and reacts to any future failure automatically — exactly what a real controller application does.

**What you are doing here**

This function has two phases:

1. install rules for the current healthy path once at startup
2. keep looping forever: sleep, check whether the current path has failed, and reroute if needed

You already wrote the hard pieces in parts 1 to 3. This final step is about connecting them so the notebook keeps using the **current** path, not the old one.

**Goal**

Install the current path, poll every few seconds, and reroute automatically when a failure appears.

**Key step**

After calling `reroute_once(...)`, the returned path must be saved back into `path_links`. If you do not update `path_links`, the loop will keep watching the old broken path and think the link is still down on every iteration.

**Hints**

- revisit `reroute_once(...)` from part 3: it returns the path the notebook should watch next
- the line to change is inside the `if detect_failure(...)` branch
- if your app keeps printing `Failure detected. Rerouting...` over and over after one failure, that usually means `path_links` was never updated

In [ ]:
def monitor_and_reroute(src_ip, dst_ip, base, auth, app_id, poll_interval=5):
    device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
    remove_rules_by_app_id(device_ids, base, auth, app_id)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    path_links = get_path(src_device, dst_device, base, auth)

    if not path_links:
        print('No path found.')
        return

    install_path_rules(
        path_links,
        src_ip,
        dst_ip,
        src_host_port,
        dst_host_port,
        base,
        auth,
        app_id,
    )
    print(f'Installed rules for a path with {len(path_links)} link(s).')

    while True:
        time.sleep(poll_interval)

        if detect_failure(path_links, base, auth):
            print('Failure detected. Rerouting...')
            new_path_links = reroute_once(path_links, src_ip, dst_ip, base, auth, app_id)
            # TODO: save the returned path back into path_links so future checks
            # watch the new route instead of the old failed one.
            # path_links = new_path_links

            if not new_path_links:
                print('No alternate path found.')
                return

            print(f'Rerouted onto a path with {len(new_path_links)} link(s).')
        else:
            print('Path still healthy.')

<details>
<summary>Show solution</summary>

```python
def monitor_and_reroute(src_ip, dst_ip, base, auth, app_id, poll_interval=5):
    device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
    remove_rules_by_app_id(device_ids, base, auth, app_id)

    hosts = api_get('hosts', base, auth)['hosts']
    src_host = find_host_by_ip(hosts, src_ip)
    dst_host = find_host_by_ip(hosts, dst_ip)
    src_device, src_host_port = get_host_location(src_host)
    dst_device, dst_host_port = get_host_location(dst_host)
    path_links = get_path(src_device, dst_device, base, auth)

    if not path_links:
        print('No path found.')
        return

    install_path_rules(
        path_links, src_ip, dst_ip, src_host_port, dst_host_port, base, auth, app_id,
    )
    print(f'Installed rules for a path with {len(path_links)} link(s).')

    while True:
        time.sleep(poll_interval)

        if detect_failure(path_links, base, auth):
            print('Failure detected. Rerouting...')
            new_path_links = reroute_once(path_links, src_ip, dst_ip, base, auth, app_id)
            path_links = new_path_links

            if not new_path_links:
                print('No alternate path found.')
                return

            print(f'Rerouted onto a path with {len(new_path_links)} link(s).')
        else:
            print('Path still healthy.')
```

</details>

**Run the app**

Before you run the next cell, bring the `s1-s2` link back up in Mininet:

```text
link s1 s2 up
```

Before you run the long-lived app cell, run the preflight check below.

Do not start the app until the preflight check prints `[ready]`.

Then run the app cell. It will keep printing `Path still healthy.` every 5 seconds while the path is healthy — that is expected.

While it is running (you will see `[*]` next to the cell):

1. in Mininet, run `h1 ping -c 3 h2` and confirm it works
2. in Mininet, run `link s1 s2 down`
3. watch the cell output — it should print `Failure detected. Rerouting...` and then `Rerouted onto ...` within a few seconds
4. run `h1 ping -c 3 h2` again and confirm it still works

**What good output looks like**

- before the failure: `Installed rules ...` followed by repeated `Path still healthy.`
- after the failure: one `Failure detected. Rerouting...` followed by one `Rerouted onto ...`

**If something looks wrong**

- if it keeps printing `Failure detected. Rerouting...` every cycle, revisit the TODO in part 4 — the loop is probably still watching the old path
- if it prints `No alternate path found.`, check that your part 3 reroute logic works and that the topology still has another path available

**To stop the loop**: click the **■ Stop** button in the Jupyter toolbar.

**Preflight**

Run the next cell before starting the automatic loop.

It checks that ONOS currently sees the healthy direct shortest path again, not just any alternate path.

For this topology, that means the current path between `h1` and `h2` should be the one-hop `s1 -> s2` route.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'

hosts = api_get('hosts', base, auth)['hosts']
src_host = find_host_by_ip(hosts, src_ip)
dst_host = find_host_by_ip(hosts, dst_ip)

if src_host is None or dst_host is None:
    print('[not ready] ONOS does not currently see both hosts.')
else:
    src_device, _ = get_host_location(src_host)
    dst_device, _ = get_host_location(dst_host)
    path_links = get_path(src_device, dst_device, base, auth)

    if not path_links:
        print('[not ready] ONOS does not currently have a path between h1 and h2.')
    elif detect_failure(path_links, base, auth):
        print('[not ready] The current path already looks failed. Make sure `link s1 s2 up` is in effect and wait a few seconds.')
    elif len(path_links) != 1:
        print(json.dumps(path_links, indent=2))
        print('[not ready] ONOS is still using a longer alternate path. Wait until the direct shortest path returns after `link s1 s2 up`.')
    elif path_links[0]['src']['device'] != src_device or path_links[0]['dst']['device'] != dst_device:
        print(json.dumps(path_links, indent=2))
        print('[not ready] The current path is not the expected direct path between the host switches.')
    else:
        print(json.dumps(path_links, indent=2))
        print('[ready] Direct shortest path detected. You can start the automatic loop.')

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
src_ip = '10.0.0.1'
dst_ip = '10.0.0.2'
app_id = 'org.onosproject.rest'

monitor_and_reroute(src_ip, dst_ip, base, auth, app_id)

## Cleanup

Before moving on, stop the running app cell above if it is still active (`[*]`).

Click the **■ Stop** button in the Jupyter toolbar before continuing. Otherwise the loop may keep reinstalling rules while you are trying to clean up.

Then remove the rules this notebook installed and restore `fwd`.

Run the next cell.

In [ ]:
base = 'http://localhost:8181/onos/v1'
auth = ('onos', 'rocks')
app_id = 'org.onosproject.rest'

device_ids = [device['id'] for device in api_get('devices', base, auth)['devices']]
removed = remove_rules_by_app_id(device_ids, base, auth, app_id)
print(f'Removed {removed} rule(s) with appId={app_id}.')

Reactivate `fwd` from the ONOS CLI so the network is ready for the next lab:

```text
app activate org.onosproject.fwd
```

Then verify with `pingall` in Mininet.

## You did it

You built a small controller application that can:

1. inspect the active path
2. detect when that path fails
3. remove stale rules
4. install rules for a new path
5. keep watching and reroute automatically

That is a big step beyond manually calling REST APIs.

In SDN terms, the switches handled the data plane, while your notebook acted as a simple control-plane application: it observed network state, decided when the path needed to change, and programmed new forwarding rules.

That same control-plane idea will carry forward into Lab 4, where the controller manages transport slices instead of a single reroute.

In this exercise, you turned the building blocks from the walkthrough into a working control loop.